##### Library

In [ ]:
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Any, Optional

import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

try:
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        get_linear_schedule_with_warmup,
        set_seed as hf_set_seed,
    )
except Exception:
    hf_set_seed = None

from sklearn.metrics import accuracy_score, f1_score, classification_report

##### Experiment Reproducibility & Config & Dataset

In [ ]:
# -----------------------------
# 0) Reproducibility
# -----------------------------
def seed_everything(
    seed: int = 42,
    *,
    use_cuda: bool = False,             # GPU 쓸 때만 True
    deterministic: bool = True,         # 재현성 우선(느려질 수 있음)
    enforce_determinism: bool = False,  # torch.use_deterministic_algorithms까지 강제
    set_env: bool = True,               # OS/라이브러리 레벨 환경변수도 세팅
) -> None:
    """
    실무용 시드/재현성 세팅 유틸.

    - CPU-only 기본값(use_cuda=False)
    - deterministic=True: cuDNN 결정론 옵션 + benchmark off
    - enforce_determinism=True: torch.use_deterministic_algorithms(True)로 비결정론 연산을 에러로 막음
      (실험/디버깅에 유용, production에서는 보통 False)
    """

    # -----------------------------
    # 1) Python / NumPy
    # -----------------------------
    random.seed(seed)               # 파이썬 random 난수 고정
    np.random.seed(seed)            # NumPy 난수 고정

    # -----------------------------
    # 2) PyTorch (CPU)
    # -----------------------------
    torch.manual_seed(seed)         # torch CPU 난수 고정

    # -----------------------------
    # 3) HuggingFace (선택)
    # -----------------------------
    if hf_set_seed is not None:
        hf_set_seed(seed)           # HF 내부(seed + 일부 라이브러리) 정리용

    # -----------------------------
    # 4) 환경변수(선택)
    # -----------------------------
    if set_env:
        os.environ["PYTHONHASHSEED"] = str(seed)    # Python 해시 기반 연산(딕셔너리 순서 등) 안정화

        # CUDA에서 matmul 결정론 강화(AMP/TF32/커널 등에 따라 영향)
        # - 필요할 때만 켜는 게 일반적이라 use_cuda일 때만 설정 권장
        if use_cuda:
            # PyTorch 권장 설정 중 하나. 엄격/느슨 옵션이 있음.
            # ":4096:8"은 성능/호환성 밸런스, 더 엄격하게는 ":16:8" 등을 쓰기도 함.
            os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

    # -----------------------------
    # 5) GPU 설정(옵션)
    # -----------------------------
    if use_cuda and torch.cuda.is_available():
        torch.cuda.manual_seed(seed)            # 현재 GPU seed
        torch.cuda.manual_seed_all(seed)        # 멀티 GPU seed

        if deterministic:
            #  cuDNN 결정론/벤치마크 설정 (CNN 계열에서 주로 영향)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False

        # TF32는 속도 향상 대신 미세한 수치 차이를 만들 수 있어,
        # 재현성 최우선이면 끄는 경우가 많음(특히 Ampere+)
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False

    # -----------------------------
    # 6) 결정론 알고리즘 강제(옵션)
    # -----------------------------
    if enforce_determinism:
        # 비결정론 연산을 만나면 에러를 내서 "재현성 깨지는 지점"을 즉시 찾게 해줌
        # 경고만 받고 싶으면 warn_only=True를 고려
        torch.use_deterministic_algorithms(True)

        # PyTorch 일부 연산에서 경고/에러를 더 명확히 하려면 아래도 함께 쓰기도 함
        # (버전/환경에 따라 동작 차이 있음)
        # torch.set_deterministic_debug_mode("error")  # "default"|"warn"|"error"

# seed_everything(42)  # 기본이 CPU-only
# seed_everything(42, use_cuda=True, deterministic=True, enforce_determinism=False) # GPU 실험(재현성 우선)
# seed_everything(42, use_cuda=True, deterministic=True, enforce_determinism=True) # 재현성 깨지는 연산을 “잡아내고 싶을 때”



In [ ]:
# -----------------------------
# 1) Config
# -----------------------------
@dataclass  # Config라는 클래스를 dataclass로 정의 | 하이퍼파라미터/경로/플래그 관리용
class Config:
    model_name: str = "distilbert-base-uncased"  # 사용할 사전학습 모델 이름(허깅페이스 모델 ID)
    max_length: int = 256  # 토크나이저에서 max_length로 쓸 최대 토큰 길이 
    batch_size: int = 32
    lr: float = 2e-5
    weight_decay: float = 0.01  # AdamW의 weight decay 값(가중치 L2 정규화 유사), bias/LayerNorm에는 decay를 빼는 식으로 파라미터 그룹을 나누기도 함
    epochs: int = 3  # 전체 데이터셋을 몇 번 반복 학습할지
    warmup_ratio: float = 0.06  # 전체 스텝 대비 워밍업 비율. 예: 총 1000 step이면 60 step 동안 lr을 선형 증가시키고 이후 감소 스케줄.
    grad_clip: float = 1.0  # 그래디언트 클리핑 임계값, 폭주 방지

    # runtime
    use_cuda: bool = False
    fp16: bool = False  # GPU면 True 권장, mixed precision(fp16) 사용 여부
    num_workers: int = 2  # DataLoader(num_workers=...)에서 데이터 로딩 워커 프로세스 수 | CPU-only라도 워커를 늘리면 속도에 도움 될 수 있지만, 재현성/디버깅이 중요하면 0으로 두는 경우도 많음. 윈도우/WSL/노트북 환경에서 멀티프로세싱 이슈가 날 때가 있음.
    output_dir: str = "./ckpt_seqcls"  # 체크포인트/로그 저장 경로.
    use_class_weights: bool = False  # 클래스 불균형일 때 loss에 class weight를 적용할지 여부.

    @property
    def device(self) -> torch.device:
        """use_cuda 설정을 존중해서 device 결정(실무에서 강제 CPU 디버깅 가능)."""
        if self.use_cuda and torch.cuda.is_available():
            return torch.device("cuda")
        return torch.device("cpu")

    @property
    def use_fp16(self) -> bool:
        """fp16 플래그 + GPU일 때만 True (CPU에서 fp16로 터지는 사고 방지)."""
        return self.fp16 and self.device.type == "cuda"

CFG = Config(use_cuda=False, fp16=True)  # CPU면 use_fp16은 자동 False
device = CFG.device                      # torch.device('cpu')
# [GPU]
# CFG = Config(use_cuda=True, fp16=True)
# device = CFG.device                      # cuda 가능하면 cuda
# use_fp16 = CFG.use_fp16                  # cuda면 True


In [ ]:
# -----------------------------
# 2) Dataset
# -----------------------------
class TextClsDataset(Dataset):  # torch.utils.data.Dataset를 상속해서, DataLoader가 데이터를 꺼내갈 수 있는 형태로 만듭니다.
    def __init__(self, texts: List[str], labels: Optional[List[int]], tokenizer, max_length: int):
        self.texts = texts
        self.labels = labels
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self):  # 데이터셋 길이(샘플 수) | DataLoader가 epoch 길이 계산/샘플링에 사용.
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, Any]:  # 인덱스 idx에 해당하는 1개 샘플을 반환. 반환 타입은 dict (HuggingFace 모델 입력 포맷과 호환되게).
        enc = self.tok(     # encoding(인코딩) 결과 객체 | [KEY] "input_ids": 토큰 ID 리스트 | "attention_mask": 실제 토큰(1) / 패딩(0)
            self.texts[idx],
            truncation=True,    # max_length 초과 시 잘라냄(overflow 방지).
            padding=False,  # 샘플 단위 패딩을 하지 않음. | 대신 아래 collate_fn에서 배치 단위로 패딩(실무에서 흔함: 더 빠르고 유연).
            max_length=self.max_length,
            return_tensors=None,
        )
        if self.labels is not None:
            enc["labels"] = int(self.labels[idx])   # HuggingFace Trainer/모델 forward가 기대하는 키 이름이 보통 "labels"
        return enc  # 토크나이즈 결과(+ 라벨)를 dict로 반환.


def make_collate_fn(tokenizer):
    # tokenizer 정보를 캡처해서 pad_token_id를 쓸 수 있게 함
    pad_id = tokenizer.pad_token_id

    # pad_token_id가 None인 경우(아주 드묾) 대비: eos_token_id나 0으로 fallback
    if pad_id is None:
        pad_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0

    def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:     # 배치 패딩 담당 | DataLoader(dataset, batch_size=..., collate_fn=collate_fn)처럼 연결해서 씁니다.
        # padding을 batch 단위에서 처리 (실무에서 더 빠르고 유연) | batch는 Dataset의 __getitem__이 반환한 dict들이 리스트로 모인 것. | 여기서 “배치 텐서” 형태로 변환해 반환.
        # 핵심 의도: 가장 긴 샘플 길이에 맞춰 패딩(dynamic padding). | 모든 샘플을 매번 max_length까지 패딩하는 것보다 메모리/연산 절감.
        keys = batch[0].keys()  # 배치의 첫 샘플을 기준으로 어떤 키들이 있는지 확인.
        has_labels = "labels" in keys # 라벨이 들어있는 배치인지 판별(학습/평가 vs 추론).

        input_ids = [item["input_ids"] for item in batch]   # 각 샘플에서 리스트 형태의 input_ids, attention_mask만 모아 둠.
        attention_mask = [item["attention_mask"] for item in batch]
        # token_type_ids 없는 모델도 있음 (DistilBERT 등)
        token_type_ids = [item.get("token_type_ids") for item in batch] # 모델에 따라 token_type_ids가 없을 수 있으니 get()으로 안전하게. 없으면 None이 들어감.

        # pad
        max_len = max(len(x) for x in input_ids)    # 현재 배치에서 가장 긴 시퀀스 길이를 계산. | 이 길이에 맞춰 나머지를 패딩.
        def pad_1d(seq, pad_value:int = 0):   # 1차원 리스트를 max_len까지 오른쪽 패딩하는 헬퍼. |  # 일반적으로 pad token id가 0인 모델이 많지만 항상 0은 아님(BERT는 0인 경우가 흔함).| attention_mask는 패딩 부분이 0이 맞음.
            return seq + [pad_value] * (max_len - len(seq))

        input_ids = torch.tensor([pad_1d(x, pad_id) for x in input_ids], dtype=torch.long)   # 각 샘플 input_ids를 패딩 후, 2D 텐서 (batch, max_len)로 변환. | 임베딩 인덱스이므로 long 정수 타입이 필요.
        attention_mask = torch.tensor([pad_1d(x, 0) for x in attention_mask], dtype=torch.long) # 마스크도 동일 길이로 패딩. | 패딩 위치는 0, 실제 토큰 위치는 1

        batch_out = {"input_ids": input_ids, "attention_mask": attention_mask}  # 기본 출력 dict 구성.| HuggingFace 모델 forward에 그대로 넣기 좋은 구조.

        if all(x is not None for x in token_type_ids):  # 배치 내 모든 샘플이 token_type_ids를 가진 경우에만 추가. | 일부만 있으면 텐서로 만들기 애매하니(불일치) 여기서는 아예 제외.
            token_type_ids = torch.tensor([pad_1d(x, 0) for x in token_type_ids], dtype=torch.long)  # token_type_ids도 (batch, max_len)로 패딩/텐서화 후 출력에 포함.
            batch_out["token_type_ids"] = token_type_ids

        if has_labels:
            labels = torch.tensor([item["labels"] for item in batch], dtype=torch.long) # 라벨이 있으면 1D 텐서 (batch,)로 만들어 넣음. | 분류 문제(클래스 인덱스)라 long이 일반적. (회귀면 float)
            batch_out["labels"] = labels

        return batch_out    # DataLoader가 이 dict를 배치로 반환 → model(**batch_out) 형태로 바로 학습 가능.
    
    return collate_fn

##### Metrics

In [ ]:
# -----------------------------
# 3) Metrics
# -----------------------------
@torch.no_grad()    # 데코레이터 | 이 함수가 실행되는 동안 Autograd(그래디언트 추적) 를 꺼서, 메모리 사용량 감소, 속도 증가 | 평가(evaluation) 단계에서는 보통 역전파가 필요 없으니 기본적으로 붙입니다
def evaluate(model, loader) -> Dict[str, float]:  # model: 평가할 PyTorch/Transformers 모델 | loader: 배치를 제공하는 DataLoader(또는 iterable) | 반환 타입 힌트: {"accuracy": float, "macro_f1": float} 형태의 dict.
    model.eval()    # 모델을 evaluation 모드로 전환. | Dropout 비활성화, BatchNorm이 있다면 running stats 사용 등 평가용 동작으로 바뀝니다.
    all_preds, all_labels = [], []  # 전체 데이터셋에 대한 예측값/정답값을 모아서 마지막에 sklearn metric으로 한 번에 점수 계산하려는 용도.
    
    for batch in loader:    # DataLoader로부터 배치 단위로 반복.
        batch = {k: v.to(device) for k, v in batch.items()}     # 배치의 모든 텐서를 device(CPU/GPU)로 이동. | 예: input_ids, attention_mask, labels 등이 다 같이 들어있다고 가정. | 주의: 여기서 device는 함수 바깥(전역)에 정의된 변수를 사용하고 있음.
        
        # labels 분리
        labels = batch.pop("labels")    # batch에서 "labels"를 꺼내서(labels에 저장) 동시에 batch에서는 제거. | model(**batch)에 labels를 넘기지 않으려는 의도(또는 애초에 평가 시 loss 계산 불필요). pop을 쓰면 batch["labels"]는 사라지고 labels만 따로 갖게 됩니다.  

        # forward & prediction
        out = model(**batch)    # 남은 입력들(input_ids, attention_mask 등)을 키워드 인자로 풀어서 모델 호출
        preds = out.logits.argmax(dim=-1).detach().cpu().numpy()    # out.logits: 보통 shape이 [batch_size, num_classes]. | argmax(dim=-1): 마지막 축(클래스 축)에서 가장 큰 값의 인덱스 → 예측 클래스 id. |  detach(): 그래디언트 그래프에서 분리(여기서는 no_grad라 사실상 중복 안전장치). | cpu(): CPU로 이동(sklearn은 numpy 기반이라 보통 CPU에서 처리). | numpy(): numpy 배열로 변환.
        all_preds.extend(preds.tolist())    # 배치 예측값을 파이썬 리스트로 바꿔서 all_preds에 누적.
        if torch.is_tensor(labels):
            all_labels.extend(labels.detach().cpu().numpy().tolist())   # 정답 라벨도 같은 방식으로 detach → cpu → numpy → list | 전체 정답 리스트에 누적.
        else:
            all_labels.extend(np.asarray(labels).tolist())   # labels가 이미 list/np 형태인 경우

    acc = accuracy_score(all_labels, all_preds)     # sklearn의 accuracy_score로 정확도 계산. | 두 리스트의 길이가 같고, 각 원소가 클래스 id여야 정상 동작.
    f1 = f1_score(all_labels, all_preds, average="macro")   # sklearn의 f1_score. | average="macro": 클래스별 F1을 각각 계산한 뒤 단순 평균. 데이터 불균형일 때 소수 클래스 성능도 동일 비중으로 반영. | (참고) micro는 전체 TP/FP/FN 합쳐서 계산, weighted는 클래스 지원 수로 가중 평균.
    return {"accuracy": acc, "macro_f1": f1}    # 결과를 dict로 반환.

In [ ]:
# -----------------------------
# 4) Train loop
# -----------------------------
def train(                      # 학습 전체 파이프라인(토크나이저/모델/로더/학습/평가/저장)을 묶는 함수 정의.
    train_texts: List[str],     # 학습용 입력 문장 리스트. 예: ["good movie", "bad movie", ...]
    train_labels: List[int],    # 학습용 정답 레이블 리스트(정수 인덱스). 예: [1, 0, ...]
    valid_texts: List[str],     # 검증용 입력 문장 리스트.
    valid_labels: List[int],    # 검증용 정답 레이블 리스트.
    num_labels: int,            # 분류 클래스 개수. AutoModelForSequenceClassification의 최종 분류 헤드 출력 차원을 결정.
    class_weights: Optional[List[float]] = None,    # 클래스 불균형 보정을 위한 가중치(예: CrossEntropyLoss의 weight). | None이면 가중치 없이 학습 | 주의: 현재 코드 조각에서는 인자로만 받고 아직 사용은 안 함(뒤에서 loss 만들 때 써야 의미가 생김).
):
    seed_everything(42)         # 난수 고정으로 재현성 확보.
    os.makedirs(CFG.output_dir, exist_ok=True)  # 체크포인트/로그 저장할 폴더 생성. | exist_ok=True라서 이미 폴더가 있어도 에러 없이 넘어감.

    tokenizer = AutoTokenizer.from_pretrained(CFG.model_name, use_fast=True)    # 허깅페이스의 CFG.model_name(예: "distilbert-base-uncased")에 해당하는 토크나이저를 로드. | Rust 기반 “fast tokenizer” 우선 사용 → 보통 더 빠르고, offset mapping 등 기능이 풍부.
    model = AutoModelForSequenceClassification.from_pretrained(     # CFG.model_name에 해당하는 사전학습 모델 본체를 로드하고, 그 위에 분류용 헤드(classification head) 를 붙인 형태로 가져옴. 
        CFG.model_name,
        num_labels=num_labels       # 마지막 출력 로짓(logits) 차원 = num_labels | 예: 이진분류면 2, 3클래스면 3
    ).to(device)                    # 모델 파라미터를 CPU/GPU로 이동.

    train_ds = TextClsDataset(train_texts, train_labels, tokenizer, CFG.max_length)     # 사용자 정의 Dataset(TextClsDataset) 인스턴스 생성.
    valid_ds = TextClsDataset(valid_texts, valid_labels, tokenizer, CFG.max_length)     # text를 tokenizer로 인코딩 → input_ids, attention_mask(필요시 token_type_ids) 생성 | max_length 길이로 padding/truncation | 레이블을 torch.tensor로 변환

    collate_fn = make_collate_fn(tokenizer)

    train_loader = DataLoader(          # Dataset에서 배치 단위로 샘플을 뽑아 모델에 공급하는 “이터레이터” 생성.
        train_ds,
        batch_size=CFG.batch_size,      # 한 스텝에서 처리하는 샘플 수.
        shuffle=True,                   # 학습 데이터는 매 epoch마다 섞어서 일반화 성능에 유리.
        num_workers=CFG.num_workers,    # 데이터 로딩을 병렬 프로세스로 수행(속도 개선 가능). | Windows/일부 환경에서 이 값이 크면 이슈가 생기기도 해서 조정 포인트.
        collate_fn=collate_fn,          # 배치로 묶는 방식을 커스터마이즈. | 텍스트는 길이가 들쑥날쑥하기 때문에 기본 collate로는 바로 못 묶는 경우가 많아서 보통 custom collate를 둠. | 예: dict들을 key별로 stacking, padding 정리, labels 텐서화 등
        pin_memory=False,                # (주로 GPU 학습에서) CPU→GPU 전송을 빠르게 하기 위한 pinned memory 사용. | CPU만 쓰면 큰 이득은 없고, 환경에 따라 약간의 오버헤드가 생길 수 있음.
    )
    valid_loader = DataLoader(
        valid_ds,
        batch_size=CFG.batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        collate_fn=collate_fn,
        pin_memory=False,
    )

    # Optimizer
    no_decay = ["bias", "LayerNorm.weight"]     # no_decay 리스트는 weight decay를 적용하지 않을 파라미터 이름 패턴입니다. | 관례적으로: bias (편향) LayerNorm.weight (정규화 스케일 파라미터) 는 weight decay를 주지 않는 편이 안정적/표준적입니다. (정규화 파라미터까지 decay 주면 성능이 흔들리는 경우가 많음)
    params = [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],     # 파라미터 이름 n에 "bias" 또는 "LayerNorm.weight"가 포함되어 있으면 True.
         "weight_decay": CFG.weight_decay},                                                             # not any(...) → no_decay 패턴이 없는 파라미터들 | 이 그룹에만 CFG.weight_decay 적용
        {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],         # any(...) → bias/LayerNorm.weight에 해당하는 파라미터들
         "weight_decay": 0.0},                                                                          # weight_decay=0.0으로 decay를 강제로 제거
    ]                                                                                                   # 이렇게 “파라미터 그룹”을 나눠서 optimizer에 넘기면, AdamW가 그룹별로 다른 weight decay를 적용합니다.
    optimizer = torch.optim.AdamW(params, lr=CFG.lr)    # AdamW 옵티마이저 생성. | params에 파라미터 그룹(2개)을 넣었으므로 그룹별 weight_decay가 반영됨. | lr=CFG.lr은 모든 그룹에 동일하게 적용(그룹별 lr을 따로 줄 수도 있지만 여기선 동일).

    total_steps = len(train_loader) * CFG.epochs        # 전체 학습 스텝 수 = (한 epoch의 배치 수) × (epoch 수) | len(train_loader)는 보통 “배치 개수”입니다. (dataset_size / batch_size에 근사)
    warmup_steps = int(total_steps * CFG.warmup_ratio)  # 워밍업 스텝 수 = 전체 스텝의 일정 비율 | warmup_ratio=0.06이면 초반 6% 스텝 동안 lr을 선형으로 올리고 이후 스케줄을 진행
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps    # HuggingFace의 스케줄러: warmup_steps 동안 lr 선형 증가 | 이후 total_steps까지 선형 감소(보통 0으로)
    )                                                                               # 이 스케줄러는 보통 optimizer.step() 이후 scheduler.step() 순으로 호출하는 패턴이 많고, 이 코드도 뒤에서 그 순서를 따릅니다.


    # Loss (불균형이면 class weights 적용)
    if CFG.use_class_weights:   # 설정에서 use_class_weights=True이면 불균형 데이터 보정 loss를 사용.
        assert class_weights is not None, "use_class_weights=True이면 class_weights를 넣어야 합니다."   # class weight를 쓰겠다고 해놓고 class_weights=None이면 즉시 에러 내서 조용히 망하는 걸 방지.
        w = torch.tensor(class_weights, dtype=torch.float, device=device)   # 파이썬 리스트를 텐서로 변환. | device=device로 모델/배치와 같은 디바이스에 두어 연산 에러를 방지.
        criterion = nn.CrossEntropyLoss(weight=w)   # CrossEntropyLoss는 클래스별 가중치를 받을 수 있음. | weight=w는 정답 클래스에 해당하는 항이 더 크게/작게 반영되도록 만듭니다.
    else:
        criterion = nn.CrossEntropyLoss()           # 기본 CrossEntropyLoss (가중치 없음)

    scaler = torch.amp.GradScaler(enabled=(CFG.fp16 and device.type == "cuda"))     # GPU + fp16 설정일 때만 AMP 활성화. | CPU거나 fp16=False면 enabled=False라서 scaler는 “아무 것도 안 하는 래퍼”처럼 동작합니다. | 즉, 같은 코드로 CPU/GPU를 모두 커버.
    best_f1 = -1.0      # 최고 macro_f1을 추적하기 위한 초기값. f1은 보통 0~1이므로 -1로 초기화하면 첫 epoch는 무조건 갱신됨.
    best_path = os.path.join(CFG.output_dir, "best")    # 최고 모델 저장 경로(폴더). | HuggingFace save_pretrained는 파일들이 들어갈 디렉토리를 받는 형태.

    for epoch in range(1, CFG.epochs + 1):      # epoch를 1부터 시작하도록 range를 잡음(로그 출력이 직관적).
        model.train()                           # 학습 모드로 전환. | Dropout, LayerNorm/BatchNorm 동작이 학습 모드 기준으로 바뀝니다.
        total_loss = 0.0                        # epoch 평균 loss 계산을 위해 누적.

        for batch in train_loader:              # DataLoader가 배치 단위로 dict(혹은 custom collate 결과)를 반환.
            batch = {k: v.to(device) for k, v in batch.items()}     # 배치 dict의 모든 텐서를 모델과 같은 디바이스로 이동. | 일반적으로 batch는 {"input_ids": ..., "attention_mask": ..., "labels": ...} 형태.
            labels = batch.pop("labels")        # labels를 배치에서 꺼내 별도 변수로 저장. | pop은 key를 제거하므로 이후 batch에는 입력 텐서만 남습니다. | 이유: HuggingFace 모델에 labels를 넘기면 모델 내부 loss를 계산해줄 수도 있는데, 여기서는 직접 criterion으로 loss 계산하려고 labels를 빼는 설계를 택한 것.

            optimizer.zero_grad(set_to_none=True)   # 이전 스텝의 gradient 초기화. set_to_none=True는 grad를 0 텐서로 채우는 대신 None으로 만들어 메모리/속도 측면에서 이점이 있는 경우가 많습니다.

            with torch.amp.autocast(enabled=(CFG.fp16 and device.type == "cuda")):      # GPU+fp16일 때만 autocast 활성화. autocast가 켜지면 일부 연산이 fp16(또는 정책에 따라 bf16)로 수행되어 속도/메모리 이점.
                out = model(**batch)                # batch dict를 키워드 인자로 풀어서 모델 호출. 즉 model(input_ids=..., attention_mask=..., ...)와 동일. 결과 out는 보통 SequenceClassifierOutput: out.logits (필수) (설정에 따라 hidden_states/attentions 등)
                loss = criterion(out.logits, labels)    # logits shape: [batch_size, num_labels] | labels shape: [batch_size] (정수 클래스 인덱스) | CrossEntropyLoss는 내부적으로 log_softmax + nll_loss를 수행.

            scaler.scale(loss).backward()       # AMP에서 fp16은 underflow가 쉽게 나므로, loss를 스케일업한 상태로 backward를 수행해 gradient 유효숫자 문제를 줄입니다. scaler가 disabled면 사실상 loss.backward()와 같은 효과.
            scaler.unscale_(optimizer)          # gradient clipping을 하기 전에, optimizer에 연결된 gradient들을 “원래 스케일”로 되돌립니다. 이걸 안 하면 clip이 스케일된 값 기준으로 동작해서 의미가 틀어질 수 있음.
            nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)     # gradient norm이 CFG.grad_clip을 넘으면 스케일을 줄여 폭주를 방지. Transformers 파인튜닝에서 흔한 안정화 장치

            scaler.step(optimizer)              # 스케일러가 상태를 보고(overflow 감지 등) 안전하면 optimizer.step() 실행. overflow가 감지되면 step을 스킵하는 식으로 학습 안정성 확보.
            scaler.update()                     # 다음 스텝을 위한 스케일 값을 업데이트(동적으로 scale up/down).
            scheduler.step()                    # lr 스케줄을 한 스텝 진행. 이 위치(optimizer step 이후)가 일반적으로 기대되는 패턴.

            total_loss += loss.item()           # 파이썬 float로 loss 값을 꺼내 epoch 누적. | .item()은 텐서(스칼라)를 파이썬 숫자로 변환.

        metrics = evaluate(model, valid_loader) # 검증 로더로 평가 수행. evaluate 내부에서 보통: model.eval() torch.no_grad() | 전체 배치에 대해 logits → 예측 → accuracy/macro_f1 계산 를 합니다(함수 구현에 따라 다름).
        avg_loss = total_loss / max(1, len(train_loader))   # 평균 loss 계산. | max(1, len(train_loader))는 혹시라도 loader 길이가 0인 비정상 케이스에서 0으로 나누는 것 방지
        print(f"[Epoch {epoch}] loss={avg_loss:.4f} | acc={metrics['accuracy']:.4f} | macro_f1={metrics['macro_f1']:.4f}")  # epoch별 요약 로그. metrics dict에서 accuracy, macro_f1를 가져와 출력.

        # Save best
        if metrics["macro_f1"] > best_f1:       # 이번 epoch의 macro_f1이 최고 기록이면 저장.
            best_f1 = metrics["macro_f1"]
            model.save_pretrained(best_path)    # HuggingFace 방식으로 모델 가중치/설정 파일 등을 best_path 폴더에 저장.
            tokenizer.save_pretrained(best_path)    # 토크나이저 vocab/config도 함께 저장. | 추론/재현을 위해 모델만 저장하면 부족하고 tokenizer도 같이 저장하는 게 표준.
            print(f"  -> Saved best to {best_path} (macro_f1={best_f1:.4f})")   

    return best_path    # 최고 성능 모델이 저장된 경로를 반환. 이후 inference 코드에서 from_pretrained(best_path)로 바로 로드 가능.

In [ ]:
# -----------------------------
# 5) Inference
# -----------------------------
@torch.no_grad()    # 이 함수 안에서 수행되는 모든 연산에 대해 gradient(역전파용 미분값) 계산을 비활성화합니다. | 효과: 메모리 절약, 속도 향상, 그리고 추론(inference)에서 실수로 그래프가 쌓이는 것을 방지. | with torch.no_grad():를 함수 데코레이터 형태로 쓴 것
def predict(texts: List[str], model_dir: str, batch_size: int = 64):    # predict 함수 정의. | texts: 예측할 문자열 리스트. | model_dir: 저장된 모델/토크나이저가 있는 디렉토리(허깅페이스 save_pretrained로 저장한 폴더). | batch_size: 추론할 때 배치 크기(기본 64).
    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    model.eval()    # 모델을 evaluation 모드로 전환. | Dropout/BatchNorm 같은 레이어가 학습 모드와 다르게 동작하도록 함(추론 안정성).

    collate_fn = make_collate_fn(tokenizer)

    ds = TextClsDataset(texts, labels=None, tokenizer=tokenizer, max_length=CFG.max_length)     # 입력 텍스트를 Dataset 형태로 감쌉니다. | labels=None: 추론이므로 정답 라벨이 없다는 의미. | tokenizer: 위에서 로드한 토크나이저 사용. | max_length=CFG.max_length: 설정(Config)에 있는 최대 토큰 길이로 자르거나 패딩.
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)        # shuffle=False: 추론은 순서 보존이 일반적. | collate_fn=collate_fn: 배치 묶을 때 커스텀 collate 사용(토큰 텐서들을 정리/스택).
                        # 모든 샘플의 결과를 누적할 리스트.
    all_probs = []      # all_probs: 클래스별 확률 벡터(예: [0.2, 0.8]).
    all_preds = []      # all_preds: 최종 예측 클래스 인덱스(예: 1).
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}     # 배치가 딕셔너리 형태라고 가정 (input_ids, attention_mask 등). | 각 텐서를 device로 이동. 예: CPU → GPU (또는 CPU 유지)
        out = model(**batch)                                    # out은 보통 SequenceClassifierOutput이고 out.logits가 핵심
        probs = torch.softmax(out.logits, dim=-1)               # 로짓(logits: 정규화 전 점수)을 softmax로 변환해 확률로 만듭니다. | dim=-1: 마지막 차원(클래스 차원)에 대해 softmax. | shape 예: (batch_size, num_labels).
        preds = probs.argmax(dim=-1)                            # 확률이 가장 큰 클래스 인덱스를 예측값으로 선택. | 결과 shape 예: (batch_size,).
        all_probs.extend(probs.detach().cpu().numpy().tolist()) # probs를 파이썬 리스트로 변환해 누적. | 
                                                                # detach(): 그래프에서 분리(여기선 no_grad라 큰 의미는 없지만 안전/일관성).
                                                                # cpu(): CPU로 이동(넘파이 변환은 CPU 텐서 필요).
                                                                # numpy(): numpy array로 변환.
                                                                # tolist(): 파이썬 리스트(중첩 리스트)로 변환.
                                                                # extend(...): 리스트에 원소들을 펼쳐서 추가(배치 단위 결과를 전체 결과에 붙임).
        all_preds.extend(preds.detach().cpu().numpy().tolist()) # 위와 동일한 변환 흐름으로 예측 클래스 인덱스를 리스트로 누적.

    return all_preds, all_probs     # 최종적으로 (예측 클래스 리스트, 확률 벡터 리스트)를 반환.


# -----------------------------
# 6) Example usage
# -----------------------------
if __name__ == "__main__":      # 이 파일이 직접 실행될 때만 아래 코드가 실행되게 하는 파이썬 관용구. | 다른 파일에서 import하면 실행되지 않음.
    # 예시 (실무에서는 CSV 로드해서 texts/labels 만들면 됨)
    train_texts = ["i love this", "terrible product", "so good", "worst ever"]
    train_labels = [1, 0, 1, 0]
    valid_texts = ["good quality", "bad experience"]
    valid_labels = [1, 0]
    num_labels = 2

    best_dir = train(train_texts, train_labels, valid_texts, valid_labels, num_labels=num_labels)   # train(...) 함수를 호출해서 학습 수행. | 반환값 best_dir는 보통 베스트 체크포인트가 저장된 폴더 경로(이 코드 흐름상 그렇게 설계된 듯).

    preds, probs = predict(["this is amazing", "do not buy"], best_dir)     # 학습된 best 모델 디렉토리로 추론 실행. | 입력 2개 문장에 대한 예측과 확률을 받음.
    print("preds:", preds)
    print("probs:", probs)